### 1. Carregamento das bases brutas

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../data/raw")

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dfs = {
    name: pd.read_csv(RAW_PATH / filename)
    for name, filename in files.items()
}


### 2. Visao geral das tabelas

In [ ]:
summary = []

for name, df in dfs.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "total_nulls": df.isna().sum().sum(),
        "null_pct": round(df.isna().sum().sum() / df.size * 100, 2)
    })

summary_df = pd.DataFrame(summary).sort_values("rows", ascending=False)
summary_df


### 3. Amostra inicial das tabelas

In [ ]:
dfs["customers"].head()

In [ ]:
dfs["geolocation"].head()

In [ ]:
dfs["order_items"].head()

In [ ]:
dfs["order_payments"].head()

In [ ]:
dfs["order_reviews"].head()

In [ ]:
dfs["orders"].head()

In [ ]:
dfs["products"].head()

In [ ]:
dfs["sellers"].head()

In [ ]:
dfs["category_translation"].head()

### 4. Tipos de dados e completude dos campos

In [ ]:
schema = []

for name, df in dfs.items():
    for column in df.columns:
        schema.append({
            "table": name,
            "column": column,
            "dtype": df[column].dtype,
            "non_nulls": df[column].notna().sum(),
            "nulls": df[column].isna().sum(),
            "null_pct": round(df[column].isna().mean() * 100, 2),
            "unique_values": df[column].nunique(dropna=True)
        })

schema_df = pd.DataFrame(schema)
schema_df


### 5. Chaves candidatas e cardinalidade

In [ ]:
key_checks = pd.DataFrame([
    {
        "table": "customers",
        "field": "customer_id",
        "rows": len(dfs["customers"]),
        "unique_values": dfs["customers"]["customer_id"].nunique()
    },
    {
        "table": "customers",
        "field": "customer_unique_id",
        "rows": len(dfs["customers"]),
        "unique_values": dfs["customers"]["customer_unique_id"].nunique()
    },
    {
        "table": "orders",
        "field": "order_id",
        "rows": len(dfs["orders"]),
        "unique_values": dfs["orders"]["order_id"].nunique()
    },
    {
        "table": "products",
        "field": "product_id",
        "rows": len(dfs["products"]),
        "unique_values": dfs["products"]["product_id"].nunique()
    },
    {
        "table": "sellers",
        "field": "seller_id",
        "rows": len(dfs["sellers"]),
        "unique_values": dfs["sellers"]["seller_id"].nunique()
    }
])

key_checks


### 6. Valores categoricos principais

In [ ]:
categorical_checks = {
    "orders.order_status": dfs["orders"]["order_status"],
    "payments.payment_type": dfs["order_payments"]["payment_type"],
    "reviews.review_score": dfs["order_reviews"]["review_score"],
    "products.product_category_name": dfs["products"]["product_category_name"],
    "customers.customer_state": dfs["customers"]["customer_state"],
    "sellers.seller_state": dfs["sellers"]["seller_state"]
}

for name, series in categorical_checks.items():
    print(f"\n{name}")
    display(series.value_counts(dropna=False).head(10).to_frame("rows"))


### 7. Intervalo das datas

In [ ]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"]
}

date_ranges = []

for table, columns in date_columns.items():
    for column in columns:
        values = pd.to_datetime(dfs[table][column], errors="coerce")
        date_ranges.append({
            "table": table,
            "column": column,
            "min_date": values.min(),
            "max_date": values.max(),
            "invalid_or_null": values.isna().sum()
        })

pd.DataFrame(date_ranges)


### 8. Observacoes iniciais

- A tabela `geolocation` possui muitas linhas duplicadas e mais de uma coordenada para alguns prefixos de CEP. Caso seja usada, precisará de uma regra de consolidacao.
- A tabela `order_reviews` tem muitos valores nulos nos campos de comentario. Para analises quantitativas de satisfacao, `review_score` e o campo mais consistente.
- A tabela `order_payments` pode ter mais de uma linha para o mesmo pedido. Analises por pedido devem agregar `payment_value` por `order_id`.
- Na tabela `products`, os campos `product_name_lenght` e `product_description_lenght` representam o tamanho dos textos cadastrados, nao os textos em si.
- Na tabela `orders`, alguns campos de data possuem nulos esperados para pedidos nao concluidos ou com status intermediario.
